# Создание приложений для генерации изображений (Azure OpenAI)

Есть нечто большее в LLM, чем генерация текста. Вы также можете создавать изображения по текстовым описаниям. Изображения как модальность полезны в Медтехе, архитектуре, туризме, разработке игр, маркетинге и не только. В этом уроке мы работаем с нынешними **GPT Image** моделями на Microsoft Foundry.

## Цели обучения

К концу этого урока вы сможете:

- Объяснить, что такое генерация изображений и где она полезна.
- Понять семейство моделей `gpt-image` и чем они отличаются от устаревших моделей DALL·E.
- Создавать приложение для генерации изображений и редактировать изображения.

## Что такое генерация изображений?

Модели генерации изображений создают изображения по текстовому описанию. Современные модели, такие как `gpt-image`, во время обучения изучают связь между текстом и изображениями, а затем пошагово преобразуют случайный шум в изображение, соответствующее вашему запросу.

Два хорошо известных семейства моделей для изображений:

- **`gpt-image` (OpenAI)** — текущее поколение, используемое в этом уроке. Поддерживает генерацию изображений из текста и редактирование изображений (дозаливка с маской).
- **Midjourney** — популярная сторонняя модель с собственным сервисом и рабочим процессом на базе Discord.

> Старые модели OpenAI для изображений — **DALL·E 2** и **DALL·E 3** — считаются устаревшими. DALL·E 3 больше недоступна для новых развертываний, а функции типа `create_variation` были только в DALL·E 2. Для новых приложений используйте модели `gpt-image`.

На Microsoft Foundry **`gpt-image-2`** — самая новая и самая мощная модель для изображений, рекомендованная по умолчанию. Модели `gpt-image-1.5` и `gpt-image-1-mini` также доступны.

> **Важно:** модели `gpt-image` возвращают сгенерированное изображение в виде **base64** (`b64_json`), а не в виде URL. Ваш код декодирует base64-строку в байты и сохраняет — URL для скачивания изображения отсутствует.


## Создание вашего первого приложения для генерации изображений

Итак, что нужно для создания приложения по генерации изображений? Вам нужны следующие библиотеки:

- **python-dotenv**, настоятельно рекомендуется использовать эту библиотеку для хранения ваших секретов в файле *.env* отдельно от кода.
- **openai**, эта библиотека используется для взаимодействия с OpenAI API.
- **pillow**, для работы с изображениями в Python.

Если вы этого ещё не сделали, следуйте инструкциям на странице [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst), чтобы создать ресурс Microsoft Foundry и модель. Выберите **gpt-image-2** как модель (последняя модель для изображений Azure OpenAI; DALL·E устарела).

1. Создайте файл *.env* со следующим содержимым:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Найдите эту информацию в портале Microsoft Foundry для вашего ресурса в разделе "Deployments".



1. Соберите перечисленные библиотеки в файл с именем *requirements.txt*, как показано ниже:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Затем создайте виртуальное окружение и установите библиотеки:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Для Windows используйте следующие команды для создания и активации виртуального окружения:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Добавьте следующий код в файл под названием *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # импортировать dotenv
    dotenv.load_dotenv()

    # настроить клиент Azure OpenAI (Microsoft Foundry).
    # Моделям изображений требуется недавняя версия API — проверьте в документации Foundry, какая версия требуется вашей модели.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Создать изображение, используя API генерации изображений
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Введите здесь текст запроса
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # например, "gpt-image-2"
        )
        # Установить директорию для сохранённого изображения
        image_dir = os.path.join(os.curdir, 'images')

        # Если директория не существует, создать её
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Инициализировать путь к изображению (обратите внимание, что тип файла должен быть png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Модели gpt-image возвращают изображение в виде base64 (b64_json), а не URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Отобразить изображение в программе просмотра изображений по умолчанию
        image = Image.open(image_path)
        image.show()

    # обработать исключения
    except BadRequestError as err:
        print(err)

    ```

Давайте разберём этот код:

- Сначала импортируем необходимые библиотеки, включая библиотеку OpenAI, библиотеку dotenv, модуль base64 и библиотеку Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Далее загружаем переменные окружения из файла *.env*.

    ```python
    # импортировать dotenv
    dotenv.load_dotenv()
    ```

- После этого настраиваем клиента Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- После этого генерируем изображение:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Введите текст вашего запроса здесь
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Модели `gpt-image` возвращают изображение как строку в формате **base64** в `data[0].b64_json`. Мы декодируем её в байты и записываем в файл — URL для скачивания отсутствует.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- В конце открываем изображение и используем стандартный просмотрщик изображений для его отображения:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Подробнее о генерации изображения

Рассмотрим параметры `images.generate`:

- **prompt** — текстовый запрос, используемый для генерации изображения. Здесь это "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size** — размер сгенерированного изображения (например, `1024x1024`, `1536x1024`, `1024x1536` или `"auto"`).
- **n** — количество генерируемых изображений. Здесь мы генерируем одно.
- **model** — имя развертывания модели для изображений (например, `gpt-image-2`).

> Модели для изображений не принимают параметр `temperature` — он предназначен для управления текстовой генерацией. Чтобы получить разнообразие, вызовите API снова; чтобы уменьшить разнообразие, сделайте ваш запрос более конкретным.

## Дополнительные возможности генерации изображений

Вы увидели, как с помощью нескольких строк Python можно сгенерировать изображение. Модели `gpt-image` также могут **редактировать** существующее изображение. Предоставив изображение, необязательную **маску** (которая отмечает область для изменения) и запрос, вы можете изменить часть изображения — например, добавить шляпу нашему зайчику.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# изменения также возвращаются в формате base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Исходное изображение содержит только кролика; финальное изображение — с шляпой на кролике.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
